<p align="center">
  <img src="Screenshot 2026-05-20 032456.png" alt="Description" width="700">
</p>


```text
[RAG Pipeline]
       │
       ├──► 1. Document Loading (get raw text from documents)
       │
       ├──► 2. Document Splitting (break text into chuncks)
       │
       ├──► 3. Embedding Creation (text to number)
       │
       ├──► 4. storing in a vector (storage)
       │
       ├──► 5. Retrieval (similar chunks)
       │
       └──► 6. Response Generation 

```


<div style="background: #1e1e2e; padding: 14px 20px; border-radius: 8px; width: fit-content; box-shadow: 0 4px 20px rgba(59, 130, 246, 0.15);">
    <h4 style="margin: 0; color: #ffffff; font-family: system-ui, sans-serif; font-size: 1.1em; font-weight: 500; letter-spacing: -0.3px;">
        Import Dependencies
    </h4>
</div>


In [1]:
# !pip install -U langchain langchain-huggingface sentence-transformers langchain-Groq langchain-community pypdf chromadb -q

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import os
from dotenv import load_dotenv

load_dotenv(override=True)

os.environ["GROQ_API_KEY"] = os.getenv('GROQ_API_KEY')
os.environ["LANGCHAIN_TRACING_V2"] = "true"               
os.environ["LANGCHAIN_API_KEY"] = os.getenv("lang_smith") 
os.environ["LANGCHAIN_PROJECT"] = "LangChain-Learning"    


d:\download_99\Anaconda\envs\llms_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<div id="01" style="background-color: #1e1e2e; border: 4px solid #3b82f6; padding: 14px 20px; border-radius: 8px; text-align: center; max-width: 80%; margin: 0 auto; box-shadow: 0 4px 12px rgba(0,0,0,0.3);">
    <h2 style="margin: 0; background: linear-gradient(to right, #60a5fa 0%, #00f2fe 100%); -webkit-background-clip: text; -webkit-text-fill-color: transparent; font-weight: 800; font-family: 'Helvetica Neue', sans-serif; font-size: 1.8em; display: inline-block;">
        Knowledge Base
    </h2>
</div>

<div style="background: #1e1e2e; padding: 14px 20px; border-radius: 8px; width: fit-content; box-shadow: 0 4px 20px rgba(59, 130, 246, 0.15);">
    <h4 style="margin: 0; color: #ffffff; font-family: system-ui, sans-serif; font-size: 1.1em; font-weight: 500; letter-spacing: -0.3px;">
        STEP 1 : Document Loading
    </h4>
</div>


In [3]:
loader = PyPDFLoader("Yousef_Koura_RAG_Knowledge_Base_v2.pdf")
docs = loader.load()
#docs
#docs[0].metadata
#docs[0].page_content

<div style="background: #1e1e2e; padding: 14px 20px; border-radius: 8px; width: fit-content; box-shadow: 0 4px 20px rgba(59, 130, 246, 0.15);">
    <h4 style="margin: 0; color: #ffffff; font-family: system-ui, sans-serif; font-size: 1.1em; font-weight: 500; letter-spacing: -0.3px;">
        STEP 2 : Document Splitting
    </h4>
</div>


In [4]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
splits = text_splitter.split_documents(docs)
#splits
#splits[0].metadata
#splits[0].page_content

<div style="background: #1e1e2e; padding: 14px 20px; border-radius: 8px; width: fit-content; box-shadow: 0 4px 20px rgba(59, 130, 246, 0.15);">
    <h4 style="margin: 0; color: #ffffff; font-family: system-ui, sans-serif; font-size: 1.1em; font-weight: 500; letter-spacing: -0.3px;">
        STEP 3 : Embedding & Vector Store and Retrieval
    </h4>
</div>


In [5]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1705.14it/s]


In [6]:
vectorstore = Chroma.from_documents(documents=splits, embedding=embedding)
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})
# result = retriever.invoke("what is yousef education")
# result

<div id="02" style="background-color: #1e1e2e; border: 4px solid #3b82f6; padding: 14px 20px; border-radius: 8px; text-align: center; max-width: 80%; margin: 0 auto; box-shadow: 0 4px 12px rgba(0,0,0,0.3);">
    <h2 style="margin: 0; background: linear-gradient(to right, #60a5fa 0%, #00f2fe 100%); -webkit-background-clip: text; -webkit-text-fill-color: transparent; font-weight: 800; font-family: 'Helvetica Neue', sans-serif; font-size: 1.8em; display: inline-block;">
        Chatbot
    </h2>
</div>

<div style="background: #1e1e2e; padding: 14px 20px; border-radius: 8px; width: fit-content; box-shadow: 0 4px 20px rgba(59, 130, 246, 0.15);">
    <h4 style="margin: 0; color: #ffffff; font-family: system-ui, sans-serif; font-size: 1.1em; font-weight: 500; letter-spacing: -0.3px;">
        STEP 4 : prompt template + model + parser
    </h4>
</div>


In [11]:
prompt_template = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template(    
                    "You are a personal AI assistant with access to Yousef's knowledge base. "
                    "Answer questions accurately based on the provided context. "
                    "If the answer requires calculating or inferring from dates or numbers in the context, do so and explain your reasoning. "
                    "If the context truly doesn't contain enough information, say 'I don't have that information in my knowledge base' — do not make up answers. "
                    "Keep responses clear and to the point."),
            
        HumanMessagePromptTemplate.from_template("""
        Context:
        {context}

        Question:
        {question}
        """)
        ]
    )
llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct")
Parser = StrOutputParser()

<div style="background: #1e1e2e; padding: 14px 20px; border-radius: 8px; width: fit-content; box-shadow: 0 4px 20px rgba(59, 130, 246, 0.15);">
    <h4 style="margin: 0; color: #ffffff; font-family: system-ui, sans-serif; font-size: 1.1em; font-weight: 500; letter-spacing: -0.3px;">
        STEP 5 : Build the Chain
    </h4>
</div>


In [12]:

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | Parser
)

<div style="background: #1e1e2e; padding: 14px 20px; border-radius: 8px; width: fit-content; box-shadow: 0 4px 20px rgba(59, 130, 246, 0.15);">
    <h4 style="margin: 0; color: #ffffff; font-family: system-ui, sans-serif; font-size: 1.1em; font-weight: 500; letter-spacing: -0.3px;">
        STEP 6 : Get a response
    </h4>
</div>


In [13]:
# Query the system
#response = rag_chain.invoke("what is yousefs education?")
#response = rag_chain.invoke("how many ai projects yousefs made")
#response = rag_chain.invoke("what Ai skills yousefs has")
response = rag_chain.invoke("list all of Yousef's work experience with their dates")
print(response)

Here are the work experiences of Yousef with their dates:

1. **Machine Learning Engineer (Remote)** 
   - Organization: PioPetro 
   - Location: Ohio, United States (Remote)
   - Dates: June 2024 - August 2024 (3 months)

2. **AI/ML Intern** 
   - Organization: Information Technology Institute (ITI) 
   - Location: Al Minufiyah, Egypt
   - Dates: July 2023 - September 2023 (3 months)

3. **AI/ML Intern** 
   - Organization: Digital HUB (D-HUB) 
   - Location: Cairo, Egypt
   - Dates: August 2023 - August 2023 (1 month)

4. **Engineering Intern** 
   - Organization: Elmarakby Steel 
   - Location: Egypt
   - Dates: February 2023 - February 2023 (1 month)
